In [4]:
import pathlib
import os
import datetime

In [47]:
import xarray

In [9]:
import metdb

In [6]:
!ls ~/prog/python_utils

anemoi				    msg_all_channels_202606251845Z.png
async_client01.py		    msg_all_channels_202606261445Z.nc
async_server01.py		    msg_all_channels_202606261445Z.png
bokeh_datashader.py		    msg_dust_202606261445Z.nc
cmd_args_template.py		    msg_dust_202606261445Z.png
cmd_args_template.py~		    nasa_gpm_download_vis.py
decorators.py			    notebooks
get_msg.py			    personal
mass_upload_dir.py		    survey1.csv
mass_upload_dir.py~		    survey_analysis
ms365_group_membership.ps	    ticketAggregator
msg_all_channels_202606091145Z.nc   torch_lightning_intro1.py
msg_all_channels_202606091145Z.png  vtkFacade.py
msg_all_channels_202606251845Z.nc   vtkStuff


In [7]:
pathlib.Path.home() / 'prog'/'python_utils'/'msg_all_channels_202606261445Z.nc'

PosixPath('/home/users/stephen.haddad/prog/python_utils/msg_all_channels_202606261445Z.nc')

In [48]:
xarray.open_dataset(pathlib.Path.home() / 'prog'/'python_utils'/'msg_all_channels_202606261445Z.nc')

<xarray.Dataset> Size: 3MB
Dimensions:                 (obs: 37756, channel: 10)
Dimensions without coordinates: obs, channel
Data variables:
    latitude                (obs) float32 151kB ...
    longitude               (obs) float32 151kB ...
    channel_wavelength_um   (channel) float32 40B ...
    channel_frequency_hz    (channel) float64 80B ...
    brightness_temperature  (obs, channel) float64 3MB ...
Attributes:
    title:          MSG SEVIRI Brightness Temperature Data
    source:         MetDB (MSGASR)
    history:        Created 2026-06-26 15:26Z
    validity_time:  2026-06-26T14:45:00Z

In [10]:
contact = 'stephen.haddad@metoffice.gov.uk'

In [15]:
dt_template_str = '{dt.year:04d}{dt.month:02d}{dt.day:02d}/{dt.hour:02d}{dt.minute:02d}Z'

In [16]:
start_dt = datetime.datetime(2026,8,11,12,0)
end_dt = datetime.datetime(2026,8,11,13,59)

In [18]:
start_str = dt_template_str.format(dt=start_dt)
end_str = dt_template_str.format(dt=end_dt)
start_str, end_str

('20260811/1200Z', '20260811/1359Z')

In [19]:
obs_raduk = metdb.obs(contact, 
                'MSGRADUK',
                keywords = [f'START TIME {start_str} ',
                            f'END TIME {end_str} '],
                elements = ['HOUR', 
                            'MINT', 
                            'LTTD', 
                            'LNGD',
                            (('CSR_RDNC', 'CSR_BRGTS_TMPR'), 12), 
                            'CLOD_TOP_TMPR', 
                            'CLOD_AMNT']
               )



In [21]:
obs_raduk

masked_array(data=[(12, 11, 43.76654052734375, 9.922160148620605, [--, --, --, 0.0009999999310821295, 0.001999999862164259, 0.009000000543892384, 0.03800000250339508, 0.03500000014901161, 0.06399999558925629, 0.07099999487400055, 0.06399999558925629, --], [--, 44.0, --, 284.1999816894531, 229.59999084472656, 242.40000915527344, 267.8000183105469, 250.1999969482422, 266.8000183105469, 262.1000061035156, 246.6999969482422, --], 220.09999084472656, 100.0),
                   (12, 11, 43.76552200317383, 9.882050514221191, [--, --, --, 0.0009999999310821295, 0.001999999862164259, 0.009999999776482582, 0.039000000804662704, 0.03500000014901161, 0.06599999964237213, 0.07400000095367432, 0.06599999964237213, --], [--, 40.5, --, 284.6000061035156, 230.5, 244.3000030517578, 269.3999938964844, 251.40000915527344, 268.3000183105469, 264.1999816894531, 248.09999084472656, --], 221.90000915527344, 100.0),
                   (12, 11, 43.76450729370117, 9.841939926147461, [--, --, --, 0.00099999993108

### Lightning Flash 
Meteosat Third Generation Lightning Flash data
http://www-mdb-apps/ssot/datatypes/MTGLIFL.html

In [22]:
metdb.subtypes.DTYPE_MAPS['MTGLIFL'] = {'FILE_NAME': 'S200',
                                        'URL': 'S200'}

In [23]:
obs_lightning = metdb.obs(contact, 'MTGLIFL',
      keywords=[f'START TIME {start_str} ',
                f'END TIME {end_str} '],
      elements=['FILE_NAME' ,'URL'])

In [26]:
type(obs_lightning), obs_lightning.shape

(numpy.ma.MaskedArray, (719,))

In [30]:
obs_lightning['FILE_NAME'][0]

np.bytes_(b'LFL_20260811120022_L2PF_OPE_20260811120000_20260811120010_N__O_0073_0001.nc')

In [33]:
import urllib

In [41]:
out_path =  pathlib.Path(os.environ['SCRATCH']) / obs_lightning['FILE_NAME'][0].decode("utf-8")

In [42]:
urllib.request.urlretrieve(obs_lightning['URL'][0].decode("utf-8"), out_path)

(PosixPath('/data/scratch/stephen.haddad/LFL_20260811120022_L2PF_OPE_20260811120000_20260811120010_N__O_0073_0001.nc'),
 <http.client.HTTPMessage at 0x14b1f13d60f0>)

In [43]:
out_path.is_file()

True

In [54]:
xarray.open_dataset(out_path)['flash_time'].values
     

array(['2026-08-11T11:59:59.892099968', '2026-08-11T11:59:59.953099904',
       '2026-08-11T12:00:00.841600000', '2026-08-11T12:00:01.837600000',
       '2026-08-11T12:00:02.152600064', '2026-08-11T12:00:02.560600064',
       '2026-08-11T12:00:02.625600000', '2026-08-11T12:00:03.424600064',
       '2026-08-11T12:00:03.593600000', '2026-08-11T12:00:05.391600000',
       '2026-08-11T12:00:05.586599936', '2026-08-11T12:00:07.295600000',
       '2026-08-11T12:00:09.096600064', '2026-08-11T11:59:59.950000000',
       '2026-08-11T12:00:00.190000000', '2026-08-11T12:00:00.298499968',
       '2026-08-11T12:00:00.395499904', '2026-08-11T12:00:00.611500032',
       '2026-08-11T12:00:01.646499968', '2026-08-11T12:00:01.827500032',
       '2026-08-11T12:00:01.967500032', '2026-08-11T12:00:02.890499968',
       '2026-08-11T12:00:04.172499968', '2026-08-11T12:00:04.513500032',
       '2026-08-11T12:00:04.841500032', '2026-08-11T12:00:04.855500032',
       '2026-08-11T12:00:05.047500032', '2026-08-11

In [32]:
str(obs_lightning['URL'][0])

"b'http://mdbapus-prod/moods/download//LFL_20260811120022_L2PF_OPE_20260811120000_20260811120010_N__O_0073_0001.nc'"

In [56]:
print('Number of files:', len(obs_lightning))
for file, url in zip(obs_lightning['FILE_NAME'][:10], obs_lightning['URL'][:10]):
    print('filename:', file, 'URL:', url)

Number of files: 719
filename: b'LFL_20260811120022_L2PF_OPE_20260811120000_20260811120010_N__O_0073_0001.nc' URL: b'http://mdbapus-prod/moods/download//LFL_20260811120022_L2PF_OPE_20260811120000_20260811120010_N__O_0073_0001.nc'
filename: b'LFL_20260811120033_L2PF_OPE_20260811120010_20260811120020_N__O_0073_0002.nc' URL: b'http://mdbapus-prod/moods/download//LFL_20260811120033_L2PF_OPE_20260811120010_20260811120020_N__O_0073_0002.nc'
filename: b'LFL_20260811120043_L2PF_OPE_20260811120020_20260811120030_N__O_0073_0003.nc' URL: b'http://mdbapus-prod/moods/download//LFL_20260811120043_L2PF_OPE_20260811120020_20260811120030_N__O_0073_0003.nc'
filename: b'LFL_20260811120052_L2PF_OPE_20260811120030_20260811120040_N__O_0073_0004.nc' URL: b'http://mdbapus-prod/moods/download//LFL_20260811120052_L2PF_OPE_20260811120030_20260811120040_N__O_0073_0004.nc'
filename: b'LFL_20260811120104_L2PF_OPE_20260811120040_20260811120050_N__O_0073_0005.nc' URL: b'http://mdbapus-prod/moods/download//LFL_2026081

###  	All Sky Radiances for FCI instrument on the MTG-I satellites, an EUMETSAT product.

http://www-mdb-apps/ssot/datatypes/FCIASR.html


In [59]:
obs_mtg_rad = metdb.obs(contact, 
                'FCIRADUK',
                keywords = [f'START TIME {start_str} ',
                            f'END TIME {end_str} '],
           elements = ['MINT', 'SCND', 'LTTD', 'LNGD',
                (('CSR_RDNC', 'CSR_BRGTS_TMPR'), 16), 'CLOD_TOP_TMPR', 'CLOD_AMNT'])

### Winds derived from satellite imagery
http://www-mdb-apps/ssot/datatypes/AMV.html


In [60]:
subtype = 'AMV'
  keywords = [f'START TIME {start_str} ',
              f'END TIME {end_str} ',
              'PLATFORM 225'],
  elements = ['HOUR', 'MINT', 'SCND', 'LTTD', 'LNGD', (('CMPT_VCTR_U_WIND', 'CMPT_VCTR_V_WIND'),4) ]
  

IndentationError: unexpected indent (3482806081.py, line 2)

In [ ]:
obs_wind = metdb.obs(contact, subtype, keywords, elements)